# Mall Customer Segmentation and PCA
**Cameron Batts**

## Problem
Compare cluster-count heuristics and examine how much information PCA retains in three customer attributes.

## Approach
Use 200 customers with age, annual income and spending score. Evaluate k=2–14 using K-Means and Ward agglomerative clustering with Calinski–Harabasz, negated Davies–Bouldin and silhouette scores. Clustering uses original feature scales; PCA uses standardized features.

## Results
The original six saved clustering searches all selected six clusters. Standardized PCA retained 44.27% of variance with one component and 77.57% with two. Agreement between internal metrics supports further exploration; it does not prove actionable customer segments.

## Limitations
The original plotting/tuning helper was absent. A replacement is now bundled with explicit random_state=42 and n_init=10; results may differ from the historical outputs. Original clustering outputs are retained and labeled, and the replacement helper is checked separately. Clustering is sensitive to feature scales, seeds and sample composition. No stability analysis, customer-level segment profiles or campaign validation was performed.

## Data
Data/mall_customers.csv contains 200 records. CustomerID and Gender are excluded from modeling. The original repository did not document a dataset source or license; provenance remains unverified.

## Verification
Replacement helper executed successfully for all six searches; historical notebook outputs retained without a full notebook rerun. See helper_validation.txt.

# Import libraries

In [3]:
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name in ('Notebook', 'Notebooks') else Path.cwd()
import sys
sys.path.insert(0, str(project_root))
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from utils_DA import (
    plot_dendrogram,
    tune_agglomerative_clustering,
    tune_k_means
)


# Read data and basic preparation

Here we read the data.

In [4]:
mall_customers = pd.read_csv(project_root / 'Data' / 'mall_customers.csv')
mall_customers.drop(['CustomerID', 'Gender'], axis=1, inplace=True)

In [5]:
mall_customers.head()

   Age  Annual Income (k$)  Spending Score (1-100)
0   19                  15                      39
1   21                  15                      81
2   20                  16                       6
3   23                  16                      77
4   31                  17                      40

In [6]:
_ = sns.pairplot(mall_customers)
plt.tight_layout()

<Figure size 750x750 with 12 Axes>

# k-means

We will use the [Calinski-Harabasz score](https://scikit-learn.org/stable/modules/clustering.html#calinski-harabasz-index_) as a heuristic to try and determine the best value for `k`.

In [7]:
k_means_tuning = tune_k_means(data=mall_customers, k=range(2, 15), n_init = 10)

ax = plt.gca()
ax.set_xlabel('k')
ax.set_ylabel('Calinski-Harabasz score')
plt.tight_layout()

print(f"Optimal k: {k_means_tuning.get('best_k')}")

Optimal k: 6


<Figure size 640x480 with 1 Axes>

In [8]:
k_means_tuning = tune_k_means(
    data=mall_customers, k=range(2, 15), score_function=davies_bouldin_score)

ax = plt.gca()
ax.set_xlabel('k')
ax.set_ylabel('Davies-Bouldin score (negated)')
plt.tight_layout()

print(f"Optimal k (Davies-Bouldin): {k_means_tuning.get('best_k')}")

Optimal k (Davies-Bouldin): 6


<Figure size 640x480 with 1 Axes>

In [9]:
k_means_tuning = tune_k_means(
    data=mall_customers, k=range(2, 15), score_function=silhouette_score)

ax = plt.gca()
ax.set_xlabel('k')
ax.set_ylabel('Silhouette score')
plt.tight_layout()

print(f"Optimal k (Silhouette): {k_means_tuning.get('best_k')}")

Optimal k (Silhouette): 6


<Figure size 640x480 with 1 Axes>

All three saved K-Means searches selected six clusters in the historical run. Internal metrics can disagree on other datasets; no disagreement appears in these saved outputs.

# Hierarchical clustering

In [10]:
agglomerative_clustering_tuning = tune_agglomerative_clustering(data=mall_customers, k=range(2, 15))

ax = plt.gca()
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Calinski-Harabasz score')
plt.tight_layout()

print(f"Optimal number of clusters: {agglomerative_clustering_tuning.get('best_k')}")

Optimal number of clusters: 6


<Figure size 640x480 with 1 Axes>

In [11]:
agglomerative_clustering_tuning = tune_agglomerative_clustering(data=mall_customers, k=range(2, 15), score_function=davies_bouldin_score)

ax = plt.gca()
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Davies-Bouldin score (negated)')
plt.tight_layout()

print(f"Optimal number of clusters (Davies-Bouldin): {agglomerative_clustering_tuning.get('best_k')}")

Optimal number of clusters (Davies-Bouldin): 6


<Figure size 640x480 with 1 Axes>

In [12]:
agglomerative_clustering_tuning = tune_agglomerative_clustering(data=mall_customers, k=range(2, 15), score_function=silhouette_score)

ax = plt.gca()
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Silhouette score')
plt.tight_layout()

print(f"Optimal number of clusters (Silhouette): {agglomerative_clustering_tuning.get('best_k')}")

Optimal number of clusters (Silhouette): 6


<Figure size 640x480 with 1 Axes>

All three saved hierarchical searches also selected six clusters. This supports further investigation, but does not establish business value or stable customer membership.

Compare feature scaling, resampled cluster membership and segment profiles before using groups in a campaign. Metric agreement alone does not establish that clusters represent durable customer behavior.

In [13]:
scaler = StandardScaler()
mall_customers_scaled = scaler.fit_transform(mall_customers)

pca_model = PCA(n_components=3)
pca_model.fit(mall_customers_scaled)

PCA(n_components=3)

The maximum number of principal components is 3, because the dataset has 3 features after dropping CustomerID and Gender: Age, Annual Income, and Spending Score. PCA can find at most as many principal components as there are features in the data, so 3 is the upper limit here.

Here, we plot the cumulative fraction of variability explained by an increasing number of principal components.

In [14]:
try:
    cumulative_variance_explained = np.cumsum(pca_model.explained_variance_ratio_)

    _ = plt.plot(range(1, mall_customers.shape[1] + 1), cumulative_variance_explained)
    ax = plt.gca()
    _ = ax.set_xlabel("Number of principal components")
    _ = ax.set_ylabel("Variability captured")
    _ = ax.set_xticks([1, 2, 3])
    plt.tight_layout()

    for index, item in enumerate(cumulative_variance_explained):
        print(f"Variance explained by {index + 1} components: {round(item * 100, 2)}%")
except NameError:
    print('The object `pca_model` does not exist!')

Variance explained by 1 components: 44.27%
Variance explained by 2 components: 77.57%
Variance explained by 3 components: 100.0%


<Figure size 640x480 with 1 Axes>

Two components retain 77.57% of standardized variance; three retain 100%. Whether the reduction is useful depends on an explicit variance target and downstream task, rather than a universal dimensionality rule.